# Semantic Space Calibration Experiments Using CLIP

This notebook demonstrates geometric calibration using CLIP's semantic embeddings from Hugging Face's transformers library instead of raw pixel space.

In [1]:
# Standard imports
import os
import logging
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

# Project imports
from utils.logging_config import setup_logging
from utils.model_utils import train_or_load_model
from utils.data_utils import load_and_split_data
from utils.utils import SemanticCompression
from calibrators.geometric_calibrators import GeometricCalibrator
from utils.metrics import CalibrationMetrics

# Initialize logging
setup_logging()
logger = logging.getLogger(__name__)

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")

2025-02-14 00:20:05.060868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-14 00:20:07.466625: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-02-14 00:20:13,365 - INFO - Loading faiss with AVX2 support.
2025-02-14 00:20:13,492 - INFO - Successfully loaded faiss with AVX2 support.
2025-02-14 00:20:21,820 - INFO - Using device: cpu


In [2]:
def prepare_data_semantic(dataset_name, random_state):
    """Load and prepare data with semantic embeddings."""
    # Load raw data
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_split_data(
        dataset_name, 
        random_state
    )
    
    # Initialize semantic compression with CLIP
    semantic_compressor = SemanticCompression(model_name="openai/clip-vit-base-patch32")
    
    # Process each dataset split
    logger.info("Converting datasets to semantic embeddings...")
    
    X_train_semantic, _ = semantic_compressor(X_train, y_train)
    logger.info(f"Training embeddings shape: {X_train_semantic.shape}")
    
    X_val_semantic, _ = semantic_compressor(X_val, y_val)
    logger.info(f"Validation embeddings shape: {X_val_semantic.shape}")
    
    X_test_semantic, _ = semantic_compressor(X_test, y_test)
    logger.info(f"Test embeddings shape: {X_test_semantic.shape}")
    
    return (
        X_train_semantic, X_val_semantic, X_test_semantic,
        y_train, y_val, y_test
    )

In [3]:
from utils.utils import Compression, StabilitySpace

def run_semantic_calibration_experiment(dataset_name, model_type="cnn", random_state=2, 
                                        compression_type="semantic", compression_param="mobilenet"):
    """
    Run calibration experiment using semantic compression within StabilitySpace.
    Handles both CNN and sklearn models properly by ensuring correct data shapes.

    Parameters:
        dataset_name (str): Name of the dataset (e.g., CIFAR10).
        model_type (str): Type of model (default is "cnn").
        random_state (int): Random seed for reproducibility.
        compression_type (str): Compression method (e.g., 'semantic', 'PCA', 'Avgpool', 'Maxpool').
        compression_param (str/int): Parameter for compression (e.g., 'mobilenet' for semantic, 2 for pooling).

    Returns:
        pd.DataFrame: A dataframe containing calibration metrics.
    """
    try:
        print("\n=== Starting Semantic Calibration Experiment ===")
        print(f"Dataset: {dataset_name}")
        print(f"Model type: {model_type}")
        print(f"Compression type: {compression_type}, Parameter: {compression_param}")

        # Load original data
        logger.info("Loading original data for model training...")
        X_train, X_val, X_test, y_train, y_val, y_test = load_and_split_data(dataset_name, random_state)

        print(f"\nData shapes after loading:")
        print(f"X_train: {X_train.shape}")
        print(f"X_val: {X_val.shape}")
        print(f"X_test: {X_test.shape}")

        # For sklearn models, reshape data before training
        if model_type not in ["cnn", "densenet", "pretrained_resnet"]:
            logger.info(f"Reshaping data for {model_type} model...")
            X_train_model = X_train.reshape(X_train.shape[0], -1)
            X_val_model = X_val.reshape(X_val.shape[0], -1)
            X_test_model = X_test.reshape(X_test.shape[0], -1)
        else:
            X_train_model = X_train
            X_val_model = X_val
            X_test_model = X_test

        # Train model on reshaped data
        logger.info(f"Training {model_type} model...")
        model = train_or_load_model(
            X_train_model, y_train,
            X_val_model, y_val,
            dataset_name=dataset_name,
            random_state=random_state,
            model_type=model_type
        )

        # Get predictions using properly shaped data
        logger.info("Getting model predictions...")
        if model_type in ["cnn", "densenet", "pretrained_resnet"]:
            features_test = model.predict(X_test_model)
            y_test_pred = np.argmax(features_test, axis=1)
        else:
            features_test = model.predict_proba(X_test_model)
            y_test_pred = model.predict(X_test_model)

        # Create compression configuration dynamically
        compression = Compression(
            compression_types=[compression_type],
            compression_params=[compression_param]
        )

        # Run calibration methods
        calibrations = {
            "faiss_exact": {"library": "faiss", "mode": "exact"},
            "fast_separation": {"library": "fast_separation", "mode": None},
        }

        results = []
        for name, config in calibrations.items():
            logger.info(f"Running {name} calibration with {compression_type} compression...")
            print(f"\n=== Starting {name} calibration ===")
            print(f"Configuration: {config}")

            try:
                # Create StabilitySpace with selected compression
                stability_space = StabilitySpace(
                    X_train=X_train,  
                    y_train=y_train,
                    compression=compression,
                    library=config["library"],
                    metric="cosine"
                )

                # Initialize calibrator with stability space
                calibrator = GeometricCalibrator(
                    model=model,
                    X_train=X_train_model,  
                    y_train=y_train,
                    stability_space=stability_space,
                    library=config["library"],
                    metric="cosine"
                )

                # Fit and calibrate
                calibrator.fit(X_val_model, y_val)  # Use properly shaped validation data
                calibrated_probs = calibrator.calibrate(X_test_model)  # Use properly shaped test data

                # Calculate metrics
                metrics = CalibrationMetrics(
                    calibrated_probs,
                    y_test_pred,
                    y_test,
                    n_bins=20
                )

                results.append({
                    "Method": f"{compression_type}_{name}",
                    **metrics.calculate_all_metrics()
                })

            except Exception as e:
                logger.error(f"Error in {name} calibration: {str(e)}")
                continue

        return pd.DataFrame(results)

    except Exception as e:
        logger.error(f"Error in experiment: {str(e)}")
        raise


In [4]:
import os
import logging

def run_experiments(
    datasets=["CIFAR10", "MNIST"],
    models=["RF", "GB", "cnn"],
    random_states=[42],  # List of random states to try
    compression_types=["semantic"],  # List of compression types
    compression_params={"semantic": "mobilenet"}  # Mapping of compression type to its parameters
):
    """
    Run calibration experiments for all dataset-model combinations, allowing multiple random states and compression methods.
    
    Parameters:
        datasets (list): List of dataset names to run experiments on.
        models (list): List of model types to use.
        random_states (list): List of random seed values for reproducibility.
        compression_types (list): List of compression methods to test.
        compression_params (dict): Dictionary mapping compression types to their parameters.
    """
    logger = logging.getLogger(__name__)
    
    for dataset in datasets:
        print(f"\nRunning semantic space calibration experiments for {dataset}")
        
        # Create base output directory for this dataset
        base_output_dir = f"output/semantic_calibration/{dataset}"
        os.makedirs(base_output_dir, exist_ok=True)
        
        for model_type in models:
            for random_state in random_states:
                for compression_type in compression_types:
                    compression_param = compression_params.get(compression_type, None)  # Get corresponding parameter
                    
                    try:
                        logger.info(f"Processing {dataset} with {model_type}, random_state={random_state}, "
                                    f"compression={compression_type}, param={compression_param}")
                        
                        # Run experiment with specified parameters
                        results_df = run_semantic_calibration_experiment(
                            dataset_name=dataset,
                            model_type=model_type,
                            random_state=random_state,
                            compression_type=compression_type,
                            compression_param=compression_param
                        )
                        
                        # Save results
                        model_output_dir = os.path.join(base_output_dir, model_type, f"rs_{random_state}", compression_type)
                        os.makedirs(model_output_dir, exist_ok=True)
                        results_file = os.path.join(model_output_dir, "results.csv")
                        
                        results_df.to_csv(results_file, index=False)
                        logger.info(f"Results saved to {results_file}")
                        
                        # Display results
                        display(results_df)
                    
                    except Exception as e:
                        logger.error(f"Error processing {dataset} with {model_type}, random_state={random_state}, "
                                     f"compression={compression_type}: {str(e)}")
                        continue


def prepare_data_semantic(dataset_name, random_state, model_type="mobilenet"):
    """
    Load and prepare data with semantic embeddings using specified model.
    
    Parameters:
        dataset_name (str): Name of the dataset to load
        random_state (int): Random seed for reproducibility
        model_type (str): Type of semantic model to use ('clip', 'efficientnet', or 'mobilenet')
    
    Returns:
        tuple: Processed data splits (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    # Load raw data
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_split_data(
        dataset_name, 
        random_state
    )
    
    # Initialize semantic compression with appropriate model
    model_configs = {
        'clip': "openai/clip-vit-base-patch32",
        'efficientnet': "tf_efficientnet_lite0",
        'mobilenet': "mobilenetv3_small_100"
    }
    
    logger.info(f"Initializing semantic compression with {model_type} model")
    semantic_compressor = SemanticCompression(
        model_type=model_type,
        model_name=model_configs.get(model_type)
    )
    
    # Process each dataset split with progress tracking
    logger.info(f"Converting datasets to {model_type} embeddings...")
    
    X_train_semantic, _ = semantic_compressor(X_train, y_train)
    logger.info(f"Training embeddings shape: {X_train_semantic.shape}")
    
    X_val_semantic, _ = semantic_compressor(X_val, y_val)
    logger.info(f"Validation embeddings shape: {X_val_semantic.shape}")
    
    X_test_semantic, _ = semantic_compressor(X_test, y_test)
    logger.info(f"Test embeddings shape: {X_test_semantic.shape}")
    
    return (
        X_train_semantic, X_val_semantic, X_test_semantic,
        y_train, y_val, y_test
    )

# Function to run all semantic model experiments
def run_all_semantic_experiments(
    datasets=["CIFAR10", "MNIST"],
    models=["RF", "GB", "cnn"],
    random_states=[2, 42, 100],
    semantic_models=["efficientnet", "clip", "mobilenet"]
):
    """
    Run experiments for all combinations of semantic models and collect results.
    
    Parameters:
        datasets (list): List of datasets to evaluate
        models (list): List of ML models to test
        random_states (list): Random seeds for reproducibility
        semantic_models (list): List of semantic models to compare
    """
    # Create summary directory for combined results
    summary_dir = "output/semantic_comparison"
    os.makedirs(summary_dir, exist_ok=True)
    
    # Store results for each semantic model
    all_results = []
    
    for semantic_model in semantic_models:
        logger.info(f"\nRunning experiments with {semantic_model} embeddings")
        
        # Run experiments for current semantic model
        run_experiments(
            datasets=datasets,
            models=models,
            random_states=random_states,
            compression_types=["semantic"],
            compression_params={"semantic": semantic_model}
        )
        
        # Collect results from all experiments for this semantic model
        model_results = []
        for dataset in datasets:
            for model in models:
                for rs in random_states:
                    results_path = f"output/semantic_calibration/{dataset}/{model}/rs_{rs}/semantic/results.csv"
                    if os.path.exists(results_path):
                        df = pd.read_csv(results_path)
                        df['Semantic_Model'] = semantic_model
                        df['Dataset'] = dataset
                        df['Model'] = model
                        df['Random_State'] = rs
                        model_results.append(df)
        
        all_results.extend(model_results)
    
    # Combine all results and generate comparison
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        
        # Save combined results
        combined_df.to_csv(f"{summary_dir}/all_semantic_models_comparison.csv", index=False)
        
        # Generate summary statistics
        summary = combined_df.groupby(['Semantic_Model', 'Dataset', 'Model']).agg({
            'ECE': ['mean', 'std'],
            'MCE': ['mean', 'std']
        }).round(4)
        
        summary.to_csv(f"{summary_dir}/semantic_models_summary.csv")
        logger.info(f"Comparison results saved to {summary_dir}")
        
        # Display summary
        display(summary)

In [ ]:
# # Run all experiments with automatic comparison
run_all_semantic_experiments(
    datasets=["CIFAR10", "MNIST"],
    models=["RF", "GB", "cnn"],
    random_states=[2, 42, 100],
    semantic_models=["mobilenet", "clip", "efficientnet"]
)

2025-02-14 00:20:53,678 - INFO - 
Running experiments with mobilenet embeddings
2025-02-14 00:20:53,682 - INFO - Processing CIFAR10 with RF, random_state=2, compression=semantic, param=mobilenet
2025-02-14 00:20:53,683 - INFO - Loading original data for model training...
2025-02-14 00:20:53,683 - INFO - Loading dataset: CIFAR10



Running semantic space calibration experiments for CIFAR10

=== Starting Semantic Calibration Experiment ===
Dataset: CIFAR10
Model type: RF
Compression type: semantic, Parameter: mobilenet
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


2025-02-14 00:21:07,951 - INFO - Combining and splitting data
2025-02-14 00:21:08,248 - INFO - Data normalization completed
2025-02-14 00:21:08,252 - INFO - Reshaping data for RF model...



Data shapes after loading:
X_train: (36000, 32, 32, 3)
X_val: (12000, 32, 32, 3)
X_test: (12000, 32, 32, 3)


2025-02-14 00:21:08,902 - INFO - Training RF model...
2025-02-14 00:21:08,910 - INFO - Training new RF model.
2025-02-14 00:21:08,911 - INFO - Model type RF
2025-02-14 00:27:10,947 - INFO - Model saved at output/CIFAR10/2/saved_models/RF_model.keras.
2025-02-14 00:27:10,949 - INFO - Getting model predictions...
2025-02-14 00:27:14,539 - INFO - 
=== Compression Initialization ===
2025-02-14 00:27:14,540 - INFO - Original compression_types: ['semantic']
2025-02-14 00:27:14,540 - INFO - Original compression_params: ['mobilenet']
2025-02-14 00:27:14,541 - INFO - Final compression_types: ['semantic']
2025-02-14 00:27:14,541 - INFO - Final compression_params: ['mobilenet']
2025-02-14 00:27:14,541 - INFO - Loading MobileNetV3 model mobilenetv3_small_100
2025-02-14 00:27:14,628 - INFO - Loading pretrained weights from Hugging Face hub (timm/mobilenetv3_small_100.lamb_in1k)


...

,Method,ECE,MCE
0,semantic_faiss_exact,0.005303,0.466667
1,semantic_fast_separation,0.015441,0.577232


2025-02-14 02:12:39,764 - INFO - Processing CIFAR10 with cnn, random_state=42, compression=semantic, param=mobilenet
2025-02-14 02:12:39,765 - INFO - Loading original data for model training...
2025-02-14 02:12:39,765 - INFO - Loading dataset: CIFAR10



=== Starting Semantic Calibration Experiment ===
Dataset: CIFAR10
Model type: cnn
Compression type: semantic, Parameter: mobilenet


2025-02-14 02:12:43,749 - INFO - Combining and splitting data
2025-02-14 02:12:44,147 - INFO - Data normalization completed
2025-02-14 02:12:44,151 - INFO - Training cnn model...
2025-02-14 02:12:44,154 - INFO - Using 45 epochs for training.
2025-02-14 02:12:44,159 - INFO - Training new cnn model.
/home/itayab/.conda/envs/import_env/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-02-14 02:12:44,268 - INFO - Model type cnn



Data shapes after loading:
X_train: (36000, 32, 32, 3)
X_val: (12000, 32, 32, 3)
X_test: (12000, 32, 32, 3)
Epoch 1/45
1125/1125 - 88s - 78ms/step - accuracy: 0.4514 - loss: 1.5142 - val_accuracy: 0.5591 - val_loss: 1.2797
Epoch 2/45
1125/1125 - 86s - 76ms/step - accuracy: 0.5874 - loss: 1.1625 - val_accuracy: 0.6126 - val_loss: 1.1485
Epoch 3/45
1125/1125 - 85s - 75ms/step - accuracy: 0.6596 - loss: 0.9639 - val_accuracy: 0.6409 - val_loss: 1.0263
Epoch 4/45
1125/1125 - 85s - 75ms/step - accuracy: 0.7147 - loss: 0.8051 - val_accuracy: 0.6467 - val_loss: 1.0360
Epoch 5/45
1125/1125 - 87s - 77ms/step - accuracy: 0.7656 - loss: 0.6625 - val_accuracy: 0.6782 - val_loss: 0.9452
Epoch 6/45
1125/1125 - 85s - 76ms/step - accuracy: 0.8119 - loss: 0.5377 - val_accuracy: 0.6643 - val_loss: 1.0103
Epoch 7/45
1125/1125 - 144s - 128ms/step - accuracy: 0.8446 - loss: 0.4451 - val_accuracy: 0.6802 - val_loss: 0.9848
Epoch 8/45
1125/1125 - 72s - 64ms/step - accuracy: 0.8732 - loss: 0.3650 - val_accur

In [6]:
# # Run all experiments
# # run_experiments(
# #     datasets=["CIFAR10", "MNIST"],
# #     models=["RF", "GB"],
# #     random_states=[2, 42, 100],  # Try multiple seeds
# #     compression_types=["semantic"],  # Try both compression types
# #     compression_params={"semantic": "efficientnet"}  # Define parameters per type
# # )
# 
# run_experiments(
#     datasets=["CIFAR10", "MNIST"],
#     models=["RF", "GB"],
#     random_states=[2, 42, 100],  # Try multiple seeds
#     compression_types=["semantic"],  # Try both compression types
#     compression_params={"semantic": "clip"}  # Define parameters per type
# )
# 
# run_experiments(
#     datasets=["CIFAR10", "MNIST"],
#     models=["RF", "GB"],
#     random_states=[2, 42, 100],  # Try multiple seeds
#     compression_types=["semantic"],  # Try both compression types
#     compression_params={"semantic": "mobilenet"}  # Define parameters per type
# )
# # Or run a specific experiment
# # results_df = run_semantic_calibration_experiment(
# #     dataset_name="CIFAR10",
# #     model_type="GB",
# #     random_state=2,
# #     compression_type="semantic",
# #     compression_param="mobilenet"
# # )


2025-02-13 16:48:41,867 - INFO - Processing CIFAR10 with RF, random_state=2, compression=semantic, param=clip
2025-02-13 16:48:41,867 - INFO - Loading original data for model training...
2025-02-13 16:48:41,867 - INFO - Loading dataset: CIFAR10



Running semantic space calibration experiments for CIFAR10

=== Starting Semantic Calibration Experiment ===
Dataset: CIFAR10
Model type: RF
Compression type: semantic, Parameter: clip


2025-02-13 16:48:42,124 - INFO - Combining and splitting data
2025-02-13 16:48:42,434 - INFO - Data normalization completed
2025-02-13 16:48:42,443 - INFO - Reshaping data for RF model...



Data shapes after loading:
X_train: (36000, 32, 32, 3)
X_val: (12000, 32, 32, 3)
X_test: (12000, 32, 32, 3)


2025-02-13 16:48:42,681 - INFO - Training RF model...
2025-02-13 16:48:42,681 - INFO - Loading pre-trained RF model from output/CIFAR10/2/saved_models\RF_model.keras.
2025-02-13 16:48:42,987 - INFO - Getting model predictions...
2025-02-13 16:48:45,413 - INFO - 
=== Compression Initialization ===
2025-02-13 16:48:45,413 - INFO - Original compression_types: ['semantic']
2025-02-13 16:48:45,413 - INFO - Original compression_params: ['clip']
2025-02-13 16:48:45,416 - INFO - Final compression_types: ['semantic']
2025-02-13 16:48:45,416 - INFO - Final compression_params: ['clip']
2025-02-13 16:48:45,416 - INFO - Loading CLIP model openai/clip-vit-base-patch32
2025-02-13 16:48:47,219 - INFO - Initialized clip model for semantic compression
2025-02-13 16:48:47,221 - INFO - Initialize Compression with ['semantic'] and ['clip']
2025-02-13 16:48:47,221 - INFO - Running faiss_exact calibration with semantic compression...
2025-02-13 16:48:47,223 - INFO - Initializing StabilitySpace with faiss lib


=== Starting faiss_exact calibration ===
Configuration: {'library': 'faiss', 'mode': 'exact'}


Processing with clip:   3%|▎         | 37/1125 [00:51<25:00,  1.38s/it]


KeyboardInterrupt: 